# Restaurant Sales Analysis Project
### NoteBook 3: Q3_Data_Cleaning

## 1. Import Libraries

In [1]:
import pandas as pd

## 2. Load Data


In [2]:
df3 = pd.read_excel("../../../Data/raw/Q3.xlsx")

## 3. Initial Data Inspection

This section presents an initial assessment of the dataset, including its structure, data types, missing values, and records with zero values in key financial fields to identify potential data quality issues before the cleaning process.

### 3.1 Verify Dataset Structure

In [3]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42902 entries, 0 to 42901
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    15426 non-null  datetime64[ns]
 1   Receipt Number          15426 non-null  object        
 2   Customer                15425 non-null  object        
 3   Invoice                 15425 non-null  object        
 4   Is Refunded             15426 non-null  float64       
 5   Order Lines/Product     42902 non-null  object        
 6   Order Lines/Quantity    42902 non-null  float64       
 7   Order Lines/Unit Price  42902 non-null  float64       
 8   Order Lines/Subtotal    42902 non-null  float64       
 9   Total                   15426 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(4)
memory usage: 3.3+ MB


### 3.2 Inspect Missing Values

In [4]:
df3.isna().sum()

Date                      27476
Receipt Number            27476
Customer                  27477
Invoice                   27477
Is Refunded               27476
Order Lines/Product           0
Order Lines/Quantity          0
Order Lines/Unit Price        0
Order Lines/Subtotal          0
Total                     27476
dtype: int64

### 3.3 Inspect transaction records for zero values in key financial fields.

In [5]:
# Count transaction records containing zero values in key financial fields.
zero_value_records = df3[
    (df3["Order Lines/Quantity"] == 0) |
    (df3["Order Lines/Subtotal"] == 0) |
    (df3["Total"] == 0) |
    (df3["Order Lines/Unit Price"] == 0)
]

print(f"Records requiring review: {len(zero_value_records)}")

Records requiring review: 34


### 3.4 Check for receipt numbers linked to multiple transaction dates.

In [6]:
df3.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(10)

Receipt Number
طلب 13220-008-23768    2
طلب 13081-001-0362     2
طلب 10619-003-1984     2
طلب 10619-003-1578     2
طلب 09997-001-22687    2
طلب 12068-001-25009    2
طلب 12707-008-5093     2
طلب 11150-002-0312     2
طلب 11865-023-21453    2
طلب 12091-034-20517    1
Name: Date, dtype: int64

## 4. Rename Columns

Column names were renamed to improve readability and simplify the analysis.

In [7]:
df3.rename(columns={
    "Date": "Date",
    "Receipt Number": "Receipt Number",
    "Customer": "Customer",
    "Invoice": "Invoice",
    "Is Refunded": "Is Refunded", 
    "Order Lines/Product": "Product",
    "Order Lines/Quantity": "Quantity",
    "Order Lines/Unit Price": "Unit_Price",
    "Order Lines/Subtotal": "Subtotal",
    "Total": "Total"
}, inplace=True)

## 5. Data Cleaning, Preprocessing, and Data Consistency Checks

This section performs the main data cleaning and preprocessing steps to improve the quality and consistency of the dataset. First, a copy of the original dataset is created to preserve the raw data. Unnecessary columns (`Invoice` and `Is Refunded`) are removed as they are not required for the analysis.

Next, forward filling is applied only to selected columns (`Date`, `Receipt Number`, `Quantity`, `Unit_Price`, and `Subtotal`) because these values are expected to remain consistent within the same transaction. The `Customer` column is intentionally excluded to avoid assigning incorrect customer names to unrelated transactions, while the `Total` column is excluded because it represents the overall transaction amount rather than individual product lines, and propagating its values could introduce inaccurate financial information. Records with a `Subtotal` value of zero were removed during the cleaning process. However, one transaction was manually reviewed and excluded from this rule because its zero `Subtotal` line was identified as a valid part of the transaction rather than a data quality issue. This exception was retained to preserve the accuracy of the dataset.

A data consistency check is then performed to identify receipt numbers associated with multiple transaction dates. The inspection revealed a small number of duplicate receipt numbers linked to different transaction times. These duplicate cases were resolved by removing the oldest transaction record for each affected receipt number while retaining the most recent one. Finally, the dataset was inspected for fully duplicated records to ensure data integrity. Any exact duplicate records were identified and removed to eliminate redundant transaction entries. A final validation check was then performed to confirm that no duplicate records remained before proceeding to the exploratory data analysis.

In [8]:
# Create a copy of the original dataset to preserve the raw data.
df3_copy = df3.copy()
df3_copy = df3_copy.drop(columns=['Invoice', 'Is Refunded'])

In [9]:
# Propagate transaction-level values to product rows within the same transaction.
columns_to_fill = [
    "Date",
    "Receipt Number",
    "Quantity",
    "Unit_Price",
    "Subtotal",

]


df3_copy[columns_to_fill] = df3_copy[columns_to_fill].ffill()

In [10]:
df3_copy.isna().sum()

Date                  0
Receipt Number        0
Customer          27477
Product               0
Quantity              0
Unit_Price            0
Subtotal              0
Total             27476
dtype: int64

In [11]:
receipt_to_keep = "طلب 10739-021-22692"

df3_copy = df3_copy[
    ~(
        (df3_copy["Subtotal"] == 0) &
        (df3_copy["Receipt Number"] != receipt_to_keep)
    )
]

In [12]:
# Check for receipt numbers linked to multiple transaction dates.
df3_copy.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(10)

Receipt Number
طلب 13220-008-23768    2
طلب 13081-001-0362     2
طلب 10619-003-1984     2
طلب 10619-003-1578     2
طلب 09997-001-22687    2
طلب 12068-001-25009    2
طلب 12707-008-5093     2
طلب 11150-002-0312     2
طلب 11865-023-21453    2
طلب 12091-034-20517    1
Name: Date, dtype: int64

In [13]:
# Function Remove the oldest transaction (based on Date) for a given Receipt Number.
def remove_oldest_receipt_record(df, receipt_number):

    # Check if the receipt number exists
    if receipt_number not in df["Receipt Number"].values:
        print(f"Receipt Number '{receipt_number}' not found.")
        return df

    # Find the oldest date for the receipt
    old_date = df.loc[
        df["Receipt Number"] == receipt_number,
        "Date"
    ].min()

    # Remove all rows belonging to the oldest transaction
    df = df[
        ~(
            (df["Receipt Number"] == receipt_number) &
            (df["Date"] == old_date)
        )
    ]

    return df

In [14]:
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 13220-008-23768")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 13081-001-0362")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 10619-003-1984")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 10619-003-1578")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 09997-001-22687")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 12068-001-25009")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 12707-008-5093")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 11150-002-0312")
df3_copy = remove_oldest_receipt_record(df3_copy, "طلب 11865-023-21453")


In [15]:
df3_copy.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(10)

Receipt Number
Order 10620-005-0005    1
طلب 12127-001-0068      1
طلب 12091-034-21814     1
طلب 12091-034-21815     1
طلب 12091-034-21951     1
طلب 12091-034-22052     1
طلب 12091-034-22140     1
طلب 12091-034-22193     1
طلب 12091-034-22195     1
طلب 12091-034-22346     1
Name: Date, dtype: int64

In [16]:
# Check for duplicate records.
print(f"Duplicate records: {df3_copy.duplicated().sum()}")

Duplicate records: 54


In [17]:
# Remove duplicate records.
df3_copy = df3_copy.drop_duplicates()

In [18]:
# Verify that no duplicate records remain.
print(f"Duplicate records after removal: {df3_copy.duplicated().sum()}")

Duplicate records after removal: 0


## 6. Export Clean Dataset

In [19]:
df3_copy.to_excel("../../../Data/Cleaned/clean_q3_data.xlsx", index=False)